# Chains

**LCEL(LangChain Expression Language)** 을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**

1. **PromptTemplate**
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합

   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현

   ```python
   model = ChatOpenAI(model="gpt-4-turbo")
   ```

3. **Output Parsers**
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동

   ```python
   output_parser = JsonOutputParser()
   ```

4. **Tools**
   - `@tool` 데코레이터로 도구 정의

   ```python
   @tool
   def search(query: str) -> str:
       ...
   ```

**체인 유형별 구현**

1. Simple Chain

   ```python
   chain = prompt | model | output_parser
   response = chain.invoke({"input": "..."})
   ```

2. Sequential Chain

   ```python
   chain = (
       {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
       | prompt2
       | model2
   )
   ```

3. Conditional Chain
   - `RunnableBranch` 사용

   ```python
   branch = RunnableBranch(
       (lambda x: x["topic"] == "math", math_chain),
       (lambda x: x["topic"] == "history", history_chain),
       default_chain
   )
   ```

**v1.2 주요 변경점**

- **Legacy Chain 클래스**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable`(LCEL)로 통합
- **에이전트 통합**: `create_agent`(LangGraph 기반)가 표준

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("openai:gpt-4.1-mini")
output_parser = StrOutputParser()

## Simple Chain

In [3]:
prompt = PromptTemplate.from_template('{country}의 수도가 어디입니까?')

chain = prompt | llm | output_parser
chain.invoke(input={'country' : '대한민국'})

'대한민국의 수도는 서울특별시입니다.'

## Sequential Chain
- 두 개 이상의 chain을 직렬로 연결
- 번역 chain 과 요약 chain을 연결하는 예제

In [4]:
prompt1 = PromptTemplate.from_template('다음 문장을 한글로 번역하세요: \n\n{eng_text}')
prompt2 = PromptTemplate.from_template('다음 문장을 한 문장으로 짧게 요약하세요:\n\n{kor_text}')

# 번역 체인
translation_chain = prompt1 | llm 
# 요약 체인
summary_chain = prompt2 | llm | output_parser

# 통합 체인
chain = translation_chain | summary_chain

sentence = '''
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.[3]
'''

print(chain.invoke(sentence))

인공지능(AI)은 인간 지능 관련 작업을 수행하는 시스템을 개발하는 학문 분야로, 학습, 추론, 자연어 처리 등 다양한 기술과 응용 분야를 포함하며, 범용 인공지능(AGI) 개발을 목표로 다양한 학문과 산업에서 연구되고 있습니다.


## Conditional Chain

In [5]:
from langchain_core.runnables import RunnableBranch

# 수학 선생님 체인
math_prompt = PromptTemplate.from_template('다음 문제를 풀어주세요. 단계적인 풀이를 수식(LaTex)과 함께 작성해주세요.\n\n{question}')
math_chain = math_prompt | llm | output_parser

# 기본 체인
default_prompt = PromptTemplate.from_template('당신은 친절하고, 감성적인 공감 능력이 좋은 챗봇입니다. 다음 질문에 답해주세요.\n\n{question}')
default_chain = default_prompt | llm | output_parser

# math_chain 선택 함수: math_chain을 사용해야 하는 경우 True 반환
def is_math_question(input_dict: dict) -> bool:
    question: str = input_dict.get('question', '')
    return '계산' in question or 'calc' in question

# 분기 체인
branch_chain = RunnableBranch(
    (is_math_question, math_chain),
    default_chain
)

# 수학 질문
print(branch_chain.invoke({'question' : '1254 * 3 + 50 이거 좀 계산해줘'}))

# 그외 질문
print(branch_chain.invoke({'question' : '나 오늘 부장님한테 깨졌어. 우울하다ㅠㅠ'}))


네! 주어진 문제는 다음과 같습니다.

\[
1254 \times 3 + 50
\]

단계별로 계산해 보겠습니다.

---

**1단계: 곱셈 계산**

\[
1254 \times 3 = ?
\]

계산해 보면,

\[
1254 \times 3 = 3762
\]

---

**2단계: 덧셈 계산**

\[
3762 + 50 = ?
\]

계산해 보면,

\[
3762 + 50 = 3812
\]

---

**최종 답:**

\[
1254 \times 3 + 50 = 3812
\]

따라서 답은 \(\boxed{3812}\)입니다.
아휴, 부장님한테 깨졌다고 하니 정말 속상했겠어요. 열심히 했는데 그런 말을 들으면 마음이 무거워지고 우울해지기 마련이죠. 그래도 그런 감정 느끼는 것도 너무 자연스러운 거니까 너무 자책하지 말아요. 부디 조금만 더 힘내고, 오늘은 자신에게 작은 위로와 휴식을 주는 시간도 가져보세요. 필요하면 언제든 이야기해줘요, 함께 들어줄게요.


## 실습: 질문 유형에 따라 다른 체인 실행하기
- 사용자의 입력이 번역 요청인지, 요약 요청인지, 일반 질문인지에 따라 서로 다른 체인을 실행한다.

In [13]:
translation_prompt = PromptTemplate.from_template("""
다음 문장을 자연스러운 한국어로 번역하세요.

문장:
{input}
""")

summary_prompt = PromptTemplate.from_template("""
다음 내용을 핵심만 1문장으로 요약하세요.

내용:
{input}
""")

qa_prompt = PromptTemplate.from_template("""
다음 질문에 초급 학습자가 이해하기 쉽게 답변하세요.

질문:
{input}
""")

In [14]:
translation_chain = translation_prompt | llm | output_parser
summary_chain = summary_prompt | llm | output_parser
qa_chain = qa_prompt | llm | output_parser

In [15]:
def is_translation_request(input_text: str) -> bool:
    keywords = ["번역", "translate", "영어로", "한국어로"]
    return any(keyword in input_text.lower() for keyword in keywords)


def is_summary_request(input_text: str) -> bool:
    keywords = ["요약", "정리", "summarize", "핵심"]
    return any(keyword in input_text.lower() for keyword in keywords)

In [16]:
router_chain = RunnableBranch(
    (is_translation_request, translation_chain),
    (is_summary_request, summary_chain),
    qa_chain
)

In [17]:
# 번역 요청
router_chain.invoke("다음 문장을 한국어로 번역해줘.: LangChain helps developers build LLM applications.")

'LangChain은 개발자들이 LLM 애플리케이션을 개발하도록 돕습니다.'

In [18]:
# 요약 요청
router_chain.invoke("""
다음 내용을 요약해줘.
                   
LangChain은 LLM 애플리케이션을 만들기 위한 프레임워크이다.
Prompt, Model, Output Parser, Retriever 등을 조합하여 복잡한 흐름을 구성할 수 있다.
LCEL을 사용하면 각 구성 요소를 파이프라인처럼 연결할 수 있다.                   
""")

'LangChain은 Prompt, Model, Output Parser, Retriever 등 구성 요소를 파이프라인처럼 연결해 복잡한 LLM 애플리케이션 흐름을 만드는 프레임워크이다.'

In [19]:
# 일반 질문
router_chain.invoke("LangChain의 LCEL이 뭐야?")

'LangChain의 LCEL은 "LangChain Extended Language"의 약자예요. 쉽게 말해서, LangChain에서 사용하는 특별한 언어 또는 코드 형식이에요. 이 LCEL을 사용하면 LangChain 프로그램을 더 쉽게 만들고 조작할 수 있어요. 초급 학습자에게는 "LangChain에서 사용하는 특별한 도구 같은 언어"라고 생각하면 돼요.'